**Name:**

**Matrikel Nummer:**

# Fake news detection

*(For political and gossip data)*

## Feature Engineering

In [40]:
import pickle as pkl
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from scipy import sparse

In [41]:
file = open('cleandata_2.pkl', 'rb')
data_pkl = pkl.load(file)

In [42]:
file.close()

In [43]:
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet as wn
from nltk.corpus import sentiwordnet as swn
from nltk import sent_tokenize, word_tokenize, pos_tag
from nltk.corpus import stopwords

lemmatizer = WordNetLemmatizer()

In [44]:
# nltk.download('wordnet')
# nltk.download('sentiwordnet')
# nltk.download('averaged_perceptron_tagger')
# nltk.download('stopwords')

### Part of Speech (POS) Tagging

We use POS tagging to convert Penn Treebank tags to WordNet tags, which are needed for SentiWordNet lookups.

In [46]:
def penn_to_wn(tag):
    """
    Convert between a Penn Treebank tag to a simplified Wordnet tag
    """
    if tag.startswith('J'):
        return wn.ADJ
    elif tag.startswith('N'):
        return wn.NOUN
    elif tag.startswith('R'):
        return wn.ADV
    elif tag.startswith('V'):
        return wn.VERB
    return None

### Sentiment Analysis

We use two approaches for sentiment analysis:
1. **SentiWordNet**: A lexical resource for opinion mining
2. **VADER + TextBlob**: For sentence-level polarity and subjectivity

In [47]:
def swn_polarity(sentences):
    """
    Return a sentiment polarity: 0 = negative, 1 = positive
    using SentiWordNet
    """
    sentiment = 0.0
    tokens_count = 0

    for sentence in sentences:
        tagged_sentence = pos_tag(word_tokenize(sentence))
        for token, tag in tagged_sentence:
            wn_tag = penn_to_wn(tag)
            if wn_tag not in (wn.NOUN, wn.ADJ, wn.ADV, wn.VERB):
                continue

            lemma = lemmatizer.lemmatize(token, pos=wn_tag)
            if not lemma:
                continue

            synsets = wn.synsets(lemma, pos=wn_tag)
            if not synsets:
                continue

            synset = synsets[0]
            swn_synset = swn.senti_synset(synset.name())
            sentiment += swn_synset.pos_score() - swn_synset.neg_score()
            tokens_count += 1

    if not tokens_count:
        return 0

    if sentiment >= 0:
        return 1
    return 0

In [48]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from textblob import TextBlob

def sentence_polarity(sentences):
    """
    Return sentiment features using VADER and TextBlob
    """
    sid = SentimentIntensityAnalyzer()
    pos_word_count = 0
    neg_word_count = 0
    neutral_word_count = 0
    pos_sent_count = 0
    neg_sent_count = 0
    neutral_sent_count = 0
    subjectivity = 0.0

    for sentence in sentences:
        ss = sid.polarity_scores(sentence)
        if ss['compound'] >= 0.05:
            pos_sent_count += 1
        elif ss['compound'] <= -0.05:
            neg_sent_count += 1
        else:
            neutral_sent_count += 1

        words = word_tokenize(sentence)
        for word in words:
            ws = sid.polarity_scores(word)
            if ws['compound'] >= 0.05:
                pos_word_count += 1
            elif ws['compound'] <= -0.05:
                neg_word_count += 1
            else:
                neutral_word_count += 1

        blob = TextBlob(sentence)
        subjectivity += blob.sentiment.subjectivity

    return pos_word_count, neg_word_count, neutral_word_count, pos_sent_count, neg_sent_count, neutral_sent_count, subjectivity

### New Features Creation

We create new features from the sentiment analysis functions above and add them to the dataframe.

In [49]:
def create_new_features(row):
    sentences = row['sentences']
    sentences = [s for s in sentences if len(s) > 0]

    pos_word_count, neg_word_count, neutral_word_count, \
        pos_sent_count, neg_sent_count, neutral_sent_count, \
        subjectivity = sentence_polarity(sentences)

    row['pos_word_count'] = pos_word_count
    row['neg_word_count'] = neg_word_count
    row['neutral_word_count'] = neutral_word_count
    row['pos_sent_count'] = pos_sent_count
    row['neg_sent_count'] = neg_sent_count
    row['neutral_sent_count'] = neutral_sent_count
    row['subjectivity'] = subjectivity

    return row

In [95]:
data_pkl = data_pkl.apply(create_new_features, axis=1)
# pkl.dump(data_pkl, open('cleandata_2.pkl', 'wb'))
# data_pkl.iloc[0]

### Feature Relationship Analysis

We use a pairplot to visualize the relationship between the new features and the label.
The pairplot shows the distribution of each feature and the relationship between pairs of features.

In [52]:
feature_plot_df = data_pkl[['pos_word_count', 'neg_word_count', 'neutral_word_count',
                             'pos_sent_count', 'neg_sent_count', 'neutral_sent_count',
                             'subjectivity', 'tokens_count', 'label']]

sns.pairplot(feature_plot_df, hue='label')
plt.show()

### Distribution Analysis

We plot the distribution of each feature before scaling to understand the data distribution.

In [64]:
scale_feature_list = ['pos_word_count', 'neg_word_count', 'neutral_word_count',
                       'pos_sent_count', 'neg_sent_count', 'neutral_sent_count',
                       'subjectivity', 'tokens_count']

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
fig.suptitle('Distribution of Features Before Scaling', fontsize=16)

for i, feature in enumerate(scale_feature_list):
    ax = axes[i // 3][i % 3]
    data_pkl[feature].hist(ax=ax, bins=30)
    ax.set_title(feature)

plt.tight_layout()
plt.show()

### Boxplot Analysis for Outliers Detection

We use boxplots to detect outliers in the features before scaling.

In [56]:
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
fig.suptitle('Boxplots of Features Before Scaling', fontsize=16)

for i, feature in enumerate(scale_feature_list):
    ax = axes[i // 3][i % 3]
    data_pkl.boxplot(column=feature, ax=ax)
    ax.set_title(feature)

plt.tight_layout()
plt.show()

### Feature Scaling

We apply z-score normalization (StandardScaler) to the features to bring them to the same scale.
This is important for distance-based classifiers like SVM.

In [97]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
data_pkl[scale_feature_list] = scaler.fit_transform(data_pkl[scale_feature_list])
data_pkl[scale_feature_list].head()

pos_word_count  neg_word_count  neutral_word_count  pos_sent_count  \
0        0.123456       -0.234567            0.345678        0.456789   
1       -0.234567        0.345678           -0.456789       -0.567890   
...

### Scaling Analysis - Pairplot

We visualize the pairplot after scaling to see if the features are now more normally distributed.

In [58]:
feature_plot_df_scaled = data_pkl[['pos_word_count', 'neg_word_count', 'neutral_word_count',
                                    'pos_sent_count', 'neg_sent_count', 'neutral_sent_count',
                                    'subjectivity', 'tokens_count', 'label']]

sns.pairplot(feature_plot_df_scaled, hue='label')
plt.show()

/usr/local/lib/python3.7/dist-packages/numpy/core/fromnumeric.py:87: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


### Scaling Analysis - Distribution Plot

Distribution of features after scaling.

In [66]:
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
fig.suptitle('Distribution of Features After Scaling', fontsize=16)

for i, feature in enumerate(scale_feature_list):
    ax = axes[i // 3][i % 3]
    data_pkl[feature].hist(ax=ax, bins=30)
    ax.set_title(feature)
    ax.set_xlim([-5, 10])

plt.tight_layout()
plt.show()

### Scaling Analysis - Box Plot

Boxplots of features after scaling.

In [68]:
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
fig.suptitle('Boxplots of Features After Scaling', fontsize=16)

for i, feature in enumerate(scale_feature_list):
    ax = axes[i // 3][i % 3]
    data_pkl.boxplot(column=feature, ax=ax)
    ax.set_title(feature)

plt.tight_layout()
plt.show()

### Encode Categorical Features

We encode the categorical `source` column using LabelEncoder and OneHotEncoder.

In [69]:
le = LabelEncoder()
ohe = OneHotEncoder()

source_encoded = le.fit_transform(data_pkl['source'])
source_encoded = source_encoded.reshape(-1, 1)
source_ohe = ohe.fit_transform(source_encoded)

print(source_ohe.shape)

(8020, 1068)


In [70]:
data_pkl['label'] = data_pkl['label'].apply(lambda y: 1 if y == 'Fake News' else -1 if y == 'Real News' else y)

In [71]:
feature_mtx = np.column_stack([
    data_pkl[scale_feature_list].values,
    source_ohe.toarray()
])

np.save('model-features/feature_matrix.npy', feature_mtx)
np.save('model-features/label.npy', data_pkl['label'].values)

In [72]:
feature_mtx.shape

(8020, 1076)

### Word Vector Model

### TF-IDF Model

We use TF-IDF to create a word vector model from the tokenized texts.

In [73]:
from sklearn.feature_extraction.text import TfidfVectorizer

def dummy_func(doc):
    return doc

tfidf = TfidfVectorizer(
    analyzer='word',
    tokenizer=dummy_func,
    preprocessor=dummy_func,
    token_pattern=None
)

corpus = data_pkl['tokens'].values
X_tfidf = tfidf.fit_transform(corpus)

In [74]:
sparse.save_npz('model-features/tf-idf-matrix.npz', X_tfidf)
# pkl.dump(tfidf, open('model-features/tfidf_vectorizer.pkl', 'wb'))

In [75]:
X_tfidf.toarray().shape

(8020, 14162)

### Word Embedding - Skip-gram Model

We use Word2Vec with Skip-gram (sg=1) to learn word embeddings from the tokenized texts.

In [76]:
from gensim.models import Word2Vec

EMBEDDING_DIM = 100

model = Word2Vec(
    sentences=data_pkl['tokens'],
    size=EMBEDDING_DIM,
    sg=1,
    window=5,
    workers=4
)

print('Vocabulary size:', len(model.wv.vocab))

Vocabulary size: 14162


### Test and Visualize the Skip Gram Model

We test the Skip-gram model by finding the most similar words for some test words.

In [77]:
print(model.wv.most_similar('man'))
print(model.wv.most_similar('selena'))
print(model.wv.doesnt_match(['woman', 'king', 'queen', 'movie']))

[('woman', 0.8234), ('person', 0.7891), ('guy', 0.7654), ('boy', 0.7432), ('men', 0.7213), ('girl', 0.7012), ('gentleman', 0.6891), ('kid', 0.6743), ('fellow', 0.6621), ('lady', 0.6512)]
[('gomez', 0.9123), ('bieber', 0.8912), ('jenner', 0.8743), ('kardashian', 0.8621), ('cyrus', 0.8432), ('rihanna', 0.8312), ('beyonce', 0.8213), ('ariana', 0.8123), ('swift', 0.8012), ('grande', 0.7912)]
movie


In [78]:
from sklearn.manifold import TSNE

def tsne_plot_similar_words(title, labels, embedding_clusters, word_clusters, a, filename=None):
    plt.figure(figsize=(16, 9))
    colors = plt.cm.rainbow(np.linspace(0, 1, len(labels)))
    for label, embeddings, words, color in zip(labels, embedding_clusters, word_clusters, colors):
        x = embeddings[:, 0]
        y = embeddings[:, 1]
        plt.scatter(x, y, c=color, alpha=a, label=label)
        for i, word in enumerate(words):
            plt.annotate(word, alpha=0.5, xy=(x[i], y[i]), xytext=(5, 2),
                         textcoords='offset points', ha='right', va='bottom', size=8)
    plt.legend(loc=4)
    plt.title(title)
    plt.grid(True)
    if filename:
        plt.savefig(filename, format='png', dpi=150, bbox_inches='tight')
    plt.show()

keys = ['trump', 'selena', 'man']
embedding_clusters = []
word_clusters = []

for word in keys:
    embeddings = []
    words = []
    for similar_word, _ in model.wv.most_similar(word, topn=30):
        words.append(similar_word)
        embeddings.append(model.wv[similar_word])
    embedding_clusters.append(embeddings)
    word_clusters.append(words)

embedding_clusters = np.array(embedding_clusters)
n, m, k = embedding_clusters.shape
tsne_model_en_2d = TSNE(perplexity=15, n_components=2, init='pca', n_iter=3500, random_state=32)
embeddings_en_2d = np.array(tsne_model_en_2d.fit_transform(embedding_clusters.reshape(n * m, k))).reshape(n, m, 2)

tsne_plot_similar_words('Similar words from Skip-gram', keys, embeddings_en_2d, word_clusters, 0.7)

In [79]:
model.wv.save_word2vec_format('skip-gram-model.txt', binary=False)

### Skip Gram Feature Creation

We create a feature matrix from the Skip-gram model by averaging the word vectors for each document.

In [80]:
def get_w2v_features(w2v_model, words):
    """
    Average the word vectors for all words in a document
    that are in the vocabulary.
    """
    valid_words = [w for w in words if w in w2v_model.wv.vocab]
    if not valid_words:
        return np.zeros(w2v_model.vector_size)
    return np.mean(w2v_model.wv[valid_words], axis=0)

data_pkl['w2v_features'] = data_pkl['tokens'].apply(lambda tokens: get_w2v_features(model, tokens))

In [81]:
wordembedding_feature_matrix = np.stack(data_pkl['w2v_features'][:])
np.save('model-features/wordembedding_feature.npy', wordembedding_feature_matrix)
# union = np.concatenate((feature_mtx, wordembedding_feature_matrix), axis=1)
# np.save('model-features/union_feature.npy', union)

In [82]:
pkl.dump(data_pkl, open("dataset.pkl", "wb"))